# 04 — Lead-time predictor

`u_{n+s} = F(u_n, s)` — the network conditioned on the forecast horizon.

The question, and the only one this rung answers: does forecasting *directly* to lead time `s`
beat `s` repeated single steps? Every other column here describes the s=1 map iterated, which
is not what the rung was trained to be good at.

In [1]:
import sys, json, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks'
                       else pathlib.Path.cwd()))
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt, torch
from l63 import ARTIFACTS, evaluate as E, plots as P
from l63.data import make_datasets
from run.report import clean, load_model, load_rows, summarise

gt = json.load(open(ARTIFACTS / 'ground_truth.json'))
S  = clean(summarise(load_rows()))
DATA = list(gt['datasets'])                      # 'ode', 'sde', 'sde015'

def num(v, w=6, p=2):
    """A ruler value, or an em dash where it is genuinely undefined."""
    if isinstance(v, dict):
        v = v.get('median')
    undefined = v is None or v != v          # None from clean(), bare NaN from the json
    return f'{"—":>{w}}' if undefined else f'{v:{w}.{p}f}'

def rng(v, p=2):
    if v is None or v.get('lo') is None or v['lo'] != v['lo']:
        return '—'
    return f"[{v['lo']:.{p}f}–{v['hi']:.{p}f}]"

print(len(S), 'model x dataset entries ·', len(DATA), 'datasets')

43 model x dataset entries · 3 datasets


## Numbers

In [2]:
s = S['04_leadtime_ode']
print('  s   direct      autoregressive   ratio')
for i in (0, 1, 3, 7, 11, 15):
    dd, aa = s['direct'][i], s['autoregressive'][i]
    print(f"{i+1:3d}   {dd:.3e}   {aa:.3e}      {dd/aa:.2f}")
print(f"\nnote: {s['note']}")

  s   direct      autoregressive   ratio
  1   1.804e+00   1.804e+00      1.00
  2   1.562e+00   4.055e+00      0.39
  4   1.555e+00   9.280e+00      0.17
  8   1.552e+00   1.145e+01      0.14
 12   1.281e+00   1.280e+01      0.10
 16   2.531e+00   1.523e+01      0.17

note: read `direct` vs `autoregressive`; every other column is the s = 1 map iterated, which is not what this rung was trained to be good at


## This model

In [3]:
KEY = 'ode'      # any of DATA
s = S['04_leadtime_' + KEY]
ref = gt['datasets'][KEY]
d = make_datasets(seed=0, kind=ref['kind'], b=ref['b'])
m, hist = load_model('04_leadtime_' + KEY + '_s' + str(s['rep_seed']))

print(f"{m.n_params:,} parameters, history {m.history}, figures show seed {s['rep_seed']}")
print()
print(f"{'ruler':16s}{'median':>8s}   range over seeds")
for k in ('horizon', 'spread', 'climate', 'climate_vs_truth', 'chaos', 'alive', 'lobe'):
    v = s[k]
    p = 0 if k == 'horizon' else 2
    print(f"  {k:14s}{num(v, 8, p)}   {rng(v, p)} over {v['n']} seeds")
print(f"\ntruth on this dataset:  climate {ref['truth_climate']:.2f}   "
      f"alive {ref['truth_alive']:.2f}   lobe {ref['truth_lobe']:.2f}   "
      f"ground truth usable {ref['floor_steps']} steps")
print(f"\nfirst steps, ||u_hat_n - u_n|| in Lorenz units:")
for i, e in enumerate(s['early'][:6], 1):
    print(f"  n={i}  {e:.3e}")

17,539 parameters, history 1, figures show seed 3

ruler             median   range over seeds
  horizon             15   [10–23] over 5 seeds
  spread               —   — over 0 seeds
  climate         111.11   [87.38–120.92] over 5 seeds
  climate_vs_truth  179.07   [140.82–194.88] over 5 seeds
  chaos           -12.18   [-15.66–-3.60] over 5 seeds
  alive             0.00   [0.00–0.34] over 5 seeds
  lobe              0.00   [0.00–0.00] over 5 seeds

truth on this dataset:  climate 0.62   alive 1.00   lobe 0.63   ground truth usable 362 steps

first steps, ||u_hat_n - u_n|| in Lorenz units:
  n=1  1.804e+00
  n=2  4.055e+00
  n=3  6.669e+00
  n=4  9.280e+00
  n=5  1.086e+01
  n=6  1.156e+01


## Figures

Banked by `run/report.py`; regenerated here from the same checkpoint so the notebook and the deck cannot disagree.

In [4]:
P.loss_figure(hist, None, n_val_traj=8); plt.show()
P.arch_figure(m.spec(), "", m.n_params, None); plt.show()
long = d.raw(m.forecast(d.eval[:gt['n_long'], :m.history], gt['long_steps']))
P.lorenz_map_figure(long, d.raw(d.eval), None); plt.show()

/var/folders/pb/rz8q2sqx26z8m1lmk12bk5zm0000gn/T/ipykernel_50208/4260566466.py:1: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  P.loss_figure(hist, None, n_val_traj=8); plt.show()
/var/folders/pb/rz8q2sqx26z8m1lmk12bk5zm0000gn/T/ipykernel_50208/4260566466.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  P.arch_figure(m.spec(), "", m.n_params, None); plt.show()


/var/folders/pb/rz8q2sqx26z8m1lmk12bk5zm0000gn/T/ipykernel_50208/4260566466.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  P.lorenz_map_figure(long, d.raw(d.eval), None); plt.show()


## Findings

_Written after reading the numbers above._

- 
- 
- 